In [ ]:
%pip install tensorflow

In [ ]:
%pip install pandas

# Import library

In [ ]:
import itertools
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

# Persiapan

In [6]:
BASE_DATA_PATH = "../../data/1_cnn_image_classification"
TRAIN_PATH = os.path.join(BASE_DATA_PATH, "seg_train", "seg_train")
TEST_PATH = os.path.join(BASE_DATA_PATH, "seg_test", "seg_test")

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",  
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",  
)

class_names = train_dataset.class_names
NUM_CLASSES = len(class_names)
print(f"List kelas: {class_names}")
print(f"Jumlah kelas: {NUM_CLASSES}")

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = (
    train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

Found 14034 files belonging to 6 classes.
Found 3000 files belonging to 6 classes.
List kelas: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Jumlah kelas: 6


# Membangun model

In [ ]:
def build_cnn_model(num_layers, filter_configs, kernel_size, pool_type, input_shape=(128, 128, 3)):
    model = models.Sequential()
    model.add(layers.InputLayer(input_shape=input_shape))
    model.add(layers.Rescaling(1.0 / 255))

    PoolingLayer = (layers.MaxPooling2D if pool_type == "max" else layers.AveragePooling2D)

    for i in range(num_layers):
        f_size = (filter_configs[i] if i < len(filter_configs) else filter_configs[-1])

        model.add(
            layers.Conv2D(
                filters=f_size,
                kernel_size=kernel_size,
                padding="same",
                activation="relu",
            )
        )
        model.add(PoolingLayer(pool_size=(2, 2)))

    model.add(layers.Flatten())
    model.add(layers.Dense(64, activation="relu"))
    model.add(layers.Dense(NUM_CLASSES, activation="softmax"))

    macro_f1 = tf.keras.metrics.F1Score(average="macro", name="macro_f1")

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy", macro_f1],
    )

    return model

# Pengujian (16 eksperimen)

Jumlah layer : 2 atau 3
Filter : (32,64,128) atau (64,128,256)
Ukuran kernel : 3 atau 5
Pooling : max atau average

In [ ]:
num_layers_options = [2, 3]
filters_options = [[32, 64, 128], [64, 128, 256]]
kernel_size_options = [(3, 3), (5, 5)]
pooling_options = ["max", "avg"]

os.makedirs("saved_models", exist_ok=True)

combinations = list(itertools.product(num_layers_options, filters_options, kernel_size_options, pooling_options))

experiment_results = []
for idx, (n_layers, filters, k_size, p_type) in enumerate(combinations, start=1):
  print("=" * 67)
  print(f"Eksperimen {idx}/16 | Layers: {n_layers} | Filters: {filters[:n_layers]} | Kernel: {k_size} | Pool: {p_type}")
  print("=" * 67)

  model = build_cnn_model(n_layers, filters, k_size, p_type)
  history = model.fit(train_dataset, validation_data=val_dataset, epochs=5, verbose=1)

  train_loss = history.history.get('loss', [])
  val_loss = history.history.get('val_loss', [])
  train_f1 = history.history.get('macro_f1', history.history.get('accuracy', []))
  val_f1_list = history.history.get('val_macro_f1', history.history.get('val_accuracy', []))

  val_f1_last = val_f1_list[-1] if len(val_f1_list) > 0 else 0
  val_acc_last = history.history.get("val_accuracy", [0])[-1]

  weight_filename = f"saved_models/model_{idx}_L{n_layers}_F{filters[0]}_K{k_size[0]}_{p_type}.weights.h5"
  model.save_weights(weight_filename)

  experiment_results.append(
      {
          "id": idx,
          "layers": n_layers,
          "filters": str(filters[:n_layers]),
          "kernel": k_size,
          "pooling": p_type,
          "val_macro_f1": val_f1_last,
          "val_accuracy": val_acc_last,
          "history_loss": train_loss,
          "history_val_loss": val_loss,
          "history_f1": train_f1,
          "history_val_f1": val_f1_list
      }
  )

# Loss

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
fig.suptitle(' Grafik Training dan Validation Loss')

for i, res in enumerate(experiment_results):
    ax = axes[i // 4, i % 4]
    epochs_range = range(1, len(res["history_loss"]) + 1)

    ax.plot(epochs_range, res["history_loss"], label='Train Loss', marker='o', color='blue')
    if res["history_val_loss"]:
        ax.plot(epochs_range, res["history_val_loss"], label='Val Loss', marker='s', color='orange')

    ax.set_title(f'Eksperimen {res["id"]}')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    if i == 0:
        ax.legend()
    ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# F1-Score

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
fig.suptitle('Grafik Training and Validation F1-Score')

for i, res in enumerate(experiment_results):
    ax = axes[i // 4, i % 4]
    epochs_range = range(1, len(res["history_f1"]) + 1)

    ax.plot(epochs_range, res["history_f1"], label='Train F1', marker='o', color='green')
    if res["history_val_f1"]:
        ax.plot(epochs_range, res["history_val_f1"], label='Val F1', marker='s', color='red')

    ax.set_title(f'Eksperimen {res["id"]}')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('F1-Score')
    if i == 0:
        ax.legend()
    ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Hasil eksperimen

In [ ]:
df_results = pd.DataFrame(experiment_results)
print("REKAP HASIL EKSPERIMEN:")
display(df_results)